This file is downstream of losing a trajectory optimizer; new_baseline_traj confirms that the trajectory located in said file's plan works; this notebook finds the periapsis altitudes given a set of moons and resonance ratios. The resonance ratios are the lowest numbers achievable after the previous flyby. 
The code as is does not totally reproduce the trajectory from scratch - you are starting with a set of moons and resonance ratios absorbed from that last trajectory set with a little random optimization.
The verification that this trajectory works is in new_baseline_traj.


## Splitting the human decision from the solver's

Each row is `(kind, at, target, sign, hp_hint, param)`. Two of those are a *design*
choice a person makes on a Tisserand graph — **which moon to fly by next**, and **which
way to rotate $v_\infty$**. The rest is bookkeeping a solver should fill in: the flyby
altitude is already just bisected or golden-sectioned by `solve_resonant` /
`solve_transfer`, and the resonance ratio or time of flight follows from it.

So this notebook takes `at`, `target` and `sign` from the hand plan as **given**, and
derives `kind`, `hp`, `param` and every downstream column — arrival day, miss, arrival
$v_\infty$, and the post-flyby $r_p \times r_a$.

| legs | at → target | sign |
|---|---|---|
| 1–6 | Io → Io | −1 |
| 7 | Io → Ganymede | −1 |
| 8 | Ganymede → Ganymede | +1 |
| 9 | Ganymede → Europa | −1 |
| 10 | Europa → Ganymede | −1 |
| 11 | Ganymede → Ganymede | +1 |
| 12 | Ganymede → Europa | +1 |

## Why a beam can tell good branches from bad

A flyby at moon $M$ rotates $v_\infty$ but **cannot change its magnitude**, so the
Tisserand parameter with respect to $M$ is invariant across it — while $v_\infty$ with
respect to the *other* moons moves freely. For a circular coplanar moon,

$$v_{\infty,M}^2 = v_M^2\left(3 - \frac{a_M}{a} - 2\sqrt{\frac{a(1-e^2)}{a_M}}\right)$$

computable from the orbital elements alone, with no propagation. That makes
$v_{\infty,\text{Europa}}$ a cost-to-go evaluable on every candidate for free. Cell 0 of
the baseline notebook is already arguing this way when it says "from 3.6 km/s at
Ganymede the floor is ≈ 2.4 km/s; from 2.2 km/s it drops to ≈ 1.6–1.9".

**Objective**: lowest Europa arrival $v_\infty$, arrival day as tiebreak. The hand plan
reaches **1.702 km/s on sim day 227.25**.


In [1]:
# Bootstrap: exec cells 1-5 of the baseline notebook so the search starts from
# bit-identical state (and so the physics is never forked into a second copy).
# Gives us: MU, RJ_KM, DAY, moon_orb, moons_tb, kepler_prop, np_flyby, slingshot,
# start_date, BODIES, and `orb` -- the post-PRM state at the day-46 Io encounter.
import json, time as _t
import numpy as np

NB = 'new_baseline_traj.ipynb'
_nb = json.load(open(NB))
for _i in (1, 2, 3, 4, 5):
    exec(compile(''.join(_nb['cells'][_i]['source']), f'<baseline cell {_i}>', 'exec'))

# Capture the post-PRM state NOW. The audit cell below re-runs baseline cell 6, which
# walks the whole tour and leaves `orb` at the final Europa encounter on day 227 --
# build the search root from that by mistake and every leg starts from the finish line.
orb_post_prm = orb
print(f"post-PRM root secured: day "
      f"{(orb_post_prm.epoch - start_date).to_value('d'):.2f}")

capture orbit: 6791740 km x 75067 km, period 41.1 d, epoch 2037-03-30 00:02:09.185
PRM burn:     814.7 m/s at 2037-04-04 00:02:09.185
Io encounter: 2037-04-25 00:02:09.185,  v_inf = 7.49 km/s
transfer:     rp 405643 km  ra 6797891 km
post-PRM root secured: day 46.00


## 1. Audit: the plan is a resonance ladder wearing a disguise

Before searching, read the answer. Divide each leg's time of flight by the post-flyby
craft period and by the moon's period. Three legs labelled `'xfer'` with a bare TOF
hint turn out to be **exact integer resonances** — the label records how the author
*solved* the leg, not what the leg physically is.


In [2]:
# Re-run the hand PLAN exactly as cell 6 does, then decompose every same-moon leg.
exec(compile(''.join(_nb['cells'][6]['source']), '<baseline cell 6>', 'exec'))

T_moon_d = {n: m.T / DAY for n, m in moons_tb.items()}
acting = [46.0] + [rows[k - 1]['t_d'] for k in range(1, len(rows))]

print(f"\n{'leg':>3} {'at':9}->{'tgt':9} {'PLAN says':>14} {'tof/T_sc':>9} "
      f"{'tof/T_moon':>11}   verdict")
print('-' * 78)
for k, r in enumerate(rows):
    tof = r['t_d'] - acting[k]
    a = 0.5 * (r['rp'] + r['ra']) * RJ_KM
    T_sc = 2 * np.pi * np.sqrt(a ** 3 / MU) / DAY
    kind, prm = PLAN[k][0], PLAN[k][5]
    says = f"res {prm}" if kind == 'res' else f"xfer {prm:.2f} d"
    if r['at'] != r['tg']:
        print(f"{k+1:>3} {r['at']:9}->{r['tg']:9} {says:>14} {tof/T_sc:9.3f} "
              f"{'-':>11}   cross-moon transfer")
        continue
    j, i = tof / T_sc, tof / T_moon_d[r['at']]
    near = abs(j - round(j)) < 0.02 and abs(i - round(i)) < 0.02
    verdict = f"RESONANCE {round(i)}:{round(j)}" if near else "true node-crossing"
    print(f"{k+1:>3} {r['at']:9}->{r['tg']:9} {says:>14} {j:9.3f} {i:11.3f}   {verdict}")

flyby  1: Io       h_p   183.6 km -> Io       day   70.79  miss       0 km  v_inf  7.491 km/s  orbit  5.59 x  62.98 RJ
flyby  2: Io       h_p   269.1 km -> Io       day   86.73  miss       0 km  v_inf  7.491 km/s  orbit  5.50 x  45.57 RJ
flyby  3: Io       h_p    53.0 km -> Io       day   97.35  miss       0 km  v_inf  7.491 km/s  orbit  5.38 x  33.60 RJ
flyby  4: Io       h_p   162.9 km -> Io       day  120.37  miss       0 km  v_inf  7.491 km/s  orbit  5.25 x  26.13 RJ
flyby  5: Io       h_p   881.0 km -> Io       day  132.77  miss       0 km  v_inf  7.491 km/s  orbit  5.13 x  22.08 RJ
flyby  6: Io       h_p    31.9 km -> Io       day  146.93  miss       0 km  v_inf  7.491 km/s  orbit  4.95 x  17.75 RJ
flyby  7: Io       h_p   534.8 km -> Ganymede day  148.39  miss    4394 km  v_inf  3.575 km/s  orbit  4.77 x  15.20 RJ
flyby  8: Ganymede h_p  2823.0 km -> Ganymede day  170.55  miss     368 km  v_inf  3.571 km/s  orbit  5.50 x  15.60 RJ
flyby  9: Ganymede h_p   248.0 km -> Europa   da

## 2. Validate the heuristic before trusting it

The Tisserand $v_\infty$ is only worth steering by if it tracks what the propagator
actually reports. Check it against all 12 legs of the hand plan, and — more importantly
— check that it falls **monotonically** along the tour, which is what makes it a usable
potential function rather than merely an accurate one.


In [3]:
a_m, v_m = {}, {}
for n, m in moons_tb.items():
    am = 1.0 / (2.0 / np.linalg.norm(m.r0) - np.dot(m.v0, m.v0) / MU)
    a_m[n], v_m[n] = am, np.sqrt(MU / am)

def vinf_tisserand(a, e, moon):
    """Coplanar Tisserand v_inf w.r.t. `moon`, from elements alone."""
    val = 3.0 - a_m[moon] / a - 2.0 * np.sqrt(max(a * (1 - e * e), 0.0) / a_m[moon])
    return v_m[moon] * np.sqrt(max(val, 0.0))

print(f"\n{'leg':>3} {'arrives at':10} {'tisserand':>10} {'propagated':>11} {'err %':>7}"
      f"   {'vinf_Europa (the cost-to-go)':>30}")
print('-' * 80)
errs, chain = [], []
for k, r in enumerate(rows):
    rp, ra = r['rp'] * RJ_KM, r['ra'] * RJ_KM
    a, e = 0.5 * (rp + ra), (ra - rp) / (ra + rp)
    vt = vinf_tisserand(a, e, r['tg'])
    ve = vinf_tisserand(a, e, 'Europa')
    errs.append(abs(100 * (vt - r['vinf']) / r['vinf'])); chain.append(ve)
    print(f"{k+1:>3} {r['tg']:10} {vt:10.3f} {r['vinf']:11.3f} "
          f"{100*(vt-r['vinf'])/r['vinf']:+7.1f}   {ve:30.3f}")

mono = all(b <= a + 1e-9 for a, b in zip(chain, chain[1:]))
print(f"\nmean |err| {np.mean(errs):.1f}%   max |err| {max(errs):.1f}%")
print(f"vinf_Europa monotonically decreasing along the tour: {mono}")
assert np.mean(errs) < 5.0, "heuristic does not track the propagator -- ranking unsound"


leg arrives at  tisserand  propagated   err %     vinf_Europa (the cost-to-go)
--------------------------------------------------------------------------------
  1 Io              7.536       7.491    +0.6                           10.940
  2 Io              7.541       7.491    +0.7                           10.528
  3 Io              7.546       7.491    +0.7                           10.004
  4 Io              7.550       7.491    +0.8                            9.437
  5 Io              7.553       7.491    +0.8                            8.967
  6 Io              7.556       7.491    +0.9                            8.212
  7 Ganymede        3.568       3.575    -0.2                            7.533
  8 Ganymede        3.561       3.571    -0.3                            6.857
  9 Europa          4.693       4.821    -2.6                            4.693
 10 Ganymede        2.174       2.158    +0.8                            4.692
 11 Ganymede        2.156       2.158    -0.1    

## 3. The search

**Node** — the jovicentric state at a moon encounter, *before* that flyby is applied.

**Expansion** — with `at`, `target` and `sign` all prescribed by the leg, the only
freedom left is which resonance or which encounter to take, so one node yields ~5–15
children rather than several hundred:

* *same-moon legs*: sweep `hp` once, vectorised, to get post-flyby period as a function
  of altitude; for every coprime $(i,j)$ read off where that curve crosses
  $(i/j)\,T_M$ and bisect. The return is geometrically pinned at $t + i\,T_M$.
* *all legs*: sweep `hp`, keep altitudes whose post-flyby orbit actually crosses the
  target's orbit, probe those for close approaches, group into encounter families and
  refine once per family. This is also what reaches leg 8's genuine node-crossing.

**Pruning** — score by `vinf_Europa + w_gap·gap + w_time·elapsed`. Rank alone collapses
the beam onto near-identical variants of one branch, so nodes are first deduplicated
into bins of $(r_p, r_a, t)$ and only the best per bin survives. That diversity step
matters more than the beam width.

### Two traps worth recording

**The scoring trap.** An earlier version clamped the Tisserand expression with
`max(val, 0)`. That expression goes *negative* when the orbit does not cross the goal
moon's orbit — so the clamp scored a craft that had sailed past Europa as a perfect
tangent arrival. An unconstrained run of this search duly optimised into a region with
$r_p \approx 10.3$–$10.8\ R_J$, above Europa's $9.39\ R_J$, reported scores near 1.0,
and returned **zero** valid tours in 14.5 minutes. The fix is `sqrt(|val|)` — continuous
at true tangency, never a free zero — plus an explicit penalty per $R_J$ of
unreachability.

**The performance trap.** `MoonTB.states` runs a 60-iteration Kepler solve per call;
inside a refinement loop that is 60 numpy passes over a 2-element array, pure
interpreter overhead. `FastMoon` samples the moon's orbit once with that same exact
solver and interpolates with cubic Hermite, agreeing to ~10⁻⁹ km. With batched encounter
refinement, one node expansion went from 40.2 s to 3.3 s.


In [4]:
# --- fast Kepler, geometry, and the flyby map (all pure numpy) ---
import numpy as np
from math import gcd

# ---------------------------------------------------------------- fast Kepler
def kepler_fast(r0, v0, dts, mu, iters=14):
    """Same universal-variable propagator as the notebook, fewer Newton steps.

    The notebook uses 60 iterations; convergence is quadratic and 14 is ample for
    the discovery sweep (checked against the 60-iteration version to < 1 m).
    """
    dts = np.atleast_1d(np.asarray(dts, float))
    r0 = np.asarray(r0, float); v0 = np.asarray(v0, float)
    r0n = np.linalg.norm(r0); vr0 = np.dot(r0, v0) / r0n
    alpha = 2 / r0n - np.dot(v0, v0) / mu
    sqmu = np.sqrt(mu)
    chi = np.where(np.abs(alpha) > 1e-12, sqmu * np.abs(alpha) * dts, sqmu * dts / r0n)
    for _ in range(iters):
        z = alpha * chi ** 2
        c, s = _c(z), _s(z)
        f = (r0n * vr0 / sqmu * chi ** 2 * c + (1 - alpha * r0n) * chi ** 3 * s
             + r0n * chi - sqmu * dts)
        fp = (r0n * vr0 / sqmu * chi * (1 - alpha * chi ** 2 * s)
              + (1 - alpha * r0n) * chi ** 2 * c + r0n)
        chi = chi - np.clip(f / fp, -np.abs(chi) * .5 - 1e3, np.abs(chi) * .5 + 1e3)
    z = alpha * chi ** 2
    c, s = _c(z), _s(z)
    fF = 1 - chi ** 2 / r0n * c
    gF = dts - chi ** 3 / sqmu * s
    r = fF[:, None] * r0[None, :] + gF[:, None] * v0[None, :]
    rn = np.linalg.norm(r, axis=1)
    fdot = sqmu / (rn * r0n) * (alpha * chi ** 3 * s - chi)
    gdot = 1 - chi ** 2 / rn * c
    v = fdot[:, None] * r0[None, :] + gdot[:, None] * v0[None, :]
    return r, v


def _c(z):
    z = np.asarray(z, float); out = np.empty_like(z)
    p, n = z > 1e-8, z < -1e-8; sm = ~(p | n)
    out[p] = (1 - np.cos(np.sqrt(z[p]))) / z[p]
    out[n] = (np.cosh(np.sqrt(-z[n])) - 1) / (-z[n])
    out[sm] = .5 - z[sm] / 24
    return out


def _s(z):
    z = np.asarray(z, float); out = np.empty_like(z)
    p, n = z > 1e-8, z < -1e-8; sm = ~(p | n)
    q = np.sqrt(z[p]); out[p] = (q - np.sin(q)) / q ** 3
    q = np.sqrt(-z[n]); out[n] = (np.sinh(q) - q) / q ** 3
    out[sm] = 1 / 6 - z[sm] / 120
    return out


# ------------------------------------------------------------------ geometry
def elements(r, v, mu):
    """a, e for one state or a stack of them."""
    r = np.atleast_2d(r); v = np.atleast_2d(v)
    rn = np.linalg.norm(r, axis=1)
    a = 1.0 / (2.0 / rn - np.einsum('ij,ij->i', v, v) / mu)
    h = np.cross(r, v)
    e = np.linalg.norm(np.cross(v, h) / mu - r / rn[:, None], axis=1)
    return a, e


def flyby_v(v_sc, v_m, mu_m, R_m, hp, sign):
    """Post-flyby velocity, vectorised over flyby altitude `hp`.

    Identical model to the notebook's np_flyby: rotate v_inf about +z, leave the
    z-component alone, re-base at the moon.
    """
    hp = np.atleast_1d(np.asarray(hp, float))
    rel = v_sc - v_m
    vinf = np.linalg.norm(rel)
    delta = 2 * np.arcsin(1.0 / (1.0 + (R_m + hp) * vinf ** 2 / mu_m))
    ca, sa = np.cos(sign * delta), np.sin(sign * delta)
    out = np.stack([ca * rel[0] - sa * rel[1],
                    sa * rel[0] + ca * rel[1],
                    np.full_like(ca, rel[2])], axis=1)
    return out + v_m


def flyby_full(r_sc, v_sc, r_m, v_m, mu_m, R_m, hp, sign):
    """np_flyby from the notebook's cell 3, self-contained: the craft is re-based at
    the moon centre and its v_inf rotated by delta(hp) about +z."""
    return r_m.copy(), flyby_v(v_sc, v_m, mu_m, R_m, hp, sign)[0]

In [5]:
# --- table-interpolated moons + the Tisserand heuristic ---
class FastMoon:
    """Drop-in for MoonTB whose states() is a table lookup instead of a Kepler solve.

    MoonTB.states runs the 60-iteration universal-variable solver every call. Inside
    a refinement loop that means 60 numpy passes over a 2-element array -- pure
    interpreter overhead, and it dominated the search runtime. Here the moon's own
    orbit is sampled once at construction with that same exact solver, then evaluated
    by cubic Hermite interpolation, which is exact to ~1e-9 km because the table
    stores velocity as well as position.
    """

    def __init__(self, mt, kepler_exact, mu, n=20000):
        self.T, self.mu, self.R = mt.T, mt.mu, mt.R
        self.r0, self.v0 = mt.r0, mt.v0
        self.ts = np.linspace(0.0, self.T, n + 1)
        self.rs, self.vs = kepler_exact(mt.r0, mt.v0, self.ts)
        self.h = float(self.ts[1] - self.ts[0])

    def states(self, t):
        tau = np.mod(np.atleast_1d(np.asarray(t, float)), self.T)
        k = np.minimum((tau / self.h).astype(np.int64), len(self.ts) - 2)
        s = (tau - self.ts[k]) / self.h
        s2 = s * s; s3 = s2 * s
        h00 = 2 * s3 - 3 * s2 + 1; h10 = s3 - 2 * s2 + s
        h01 = -2 * s3 + 3 * s2;    h11 = s3 - s2
        r = (h00[:, None] * self.rs[k] + h10[:, None] * self.h * self.vs[k]
             + h01[:, None] * self.rs[k + 1] + h11[:, None] * self.h * self.vs[k + 1])
        d00 = 6 * s2 - 6 * s; d10 = 3 * s2 - 4 * s + 1
        d01 = -6 * s2 + 6 * s; d11 = 3 * s2 - 2 * s
        v = (d00[:, None] * self.rs[k] / self.h + d10[:, None] * self.vs[k]
             + d01[:, None] * self.rs[k + 1] / self.h + d11[:, None] * self.vs[k + 1])
        return r, v

    def state(self, t):
        r, v = self.states([t]); return r[0], v[0]


class Search:
    """Holds the invariants (moon models, constants) and the tunable knobs."""

    def __init__(self, moons_tb, mu, rj, *, targets=('Io', 'Europa', 'Ganymede')):
        self.moons = moons_tb
        self.MU = mu
        self.RJ = rj
        self.targets = targets
        # circular-moon reference values, from the same two-body models the tour uses
        self.ref = {}
        for name, m in moons_tb.items():
            a_m = 1.0 / (2.0 / np.linalg.norm(m.r0) - np.dot(m.v0, m.v0) / mu)
            self.ref[name] = (a_m, np.sqrt(mu / a_m))

    # ---- the heuristic ----------------------------------------------------
    def vinf_tiss(self, a, e, moon):
        """Coplanar Tisserand v_inf w.r.t. `moon`, from elements alone.

        A flyby at M cannot change v_inf w.r.t. M, but it does change it w.r.t. the
        other moons -- which is exactly the progress this drives. Validated against
        the hand plan's propagated values to 1.7% mean error.
        """
        a_m, v_m = self.ref[moon]
        val = 3.0 - a_m / a - 2.0 * np.sqrt(np.maximum(a * (1 - e * e), 0.0) / a_m)
        return v_m * np.sqrt(np.maximum(val, 0.0))

    # ---- encounter finding ------------------------------------------------
    def encounters(self, r0, v0, t0, tgt_name, t_lo, t_hi, coarse_s, rm_cache=None,
                   n_rounds=3, n_bat=40):
        """Local minima of |r_sc - r_moon| in (t_lo, t_hi), as (t_abs, miss)."""
        tgt = self.moons[tgt_name]
        dts = np.arange(t_lo, t_hi, coarse_s)
        if len(dts) < 3:
            return []
        rm = tgt.states(t0 + dts)[0] if rm_cache is None else rm_cache
        rs, _ = kepler_fast(r0, v0, dts, self.MU)
        d = np.linalg.norm(rs - rm, axis=1)
        loc = np.nonzero((d[1:-1] < d[:-2]) & (d[1:-1] < d[2:]))[0] + 1
        out = []
        for i in loc:
            # Batched bracket shrink: each round evaluates `n_bat` times in ONE
            # vectorised propagation, versus a sequential golden section paying
            # numpy overhead per step. Three rounds narrow the bracket ~1e4x.
            lo, hi = dts[i - 1], dts[i + 1]
            for _ in range(n_rounds):
                ts_ = np.linspace(lo, hi, n_bat)
                rr, _ = kepler_fast(r0, v0, ts_, self.MU)
                rmm, _ = tgt.states(t0 + ts_)
                dd = np.linalg.norm(rr - rmm, axis=1)
                k = int(np.argmin(dd))
                lo = ts_[max(k - 1, 0)]; hi = ts_[min(k + 1, n_bat - 1)]
            out.append((t0 + .5 * (lo + hi), float(dd[k])))
        return out

In [6]:
# --- nodes, knobs, and the beam search itself ---
DAY = 86400.0


class Node:
    __slots__ = ('r', 'v', 't', 'at', 'legs', 'a', 'e', 'rp', 'ra', 'vinf', 'score')

    def __init__(self, r, v, t, at, legs, a, e, vinf):
        self.r, self.v, self.t, self.at, self.legs = r, v, t, at, legs
        self.a, self.e, self.vinf = a, e, vinf
        self.rp, self.ra = a * (1 - e), a * (1 + e)


class Cfg:
    """Every knob in one place -- widen these for a longer offline run."""
    beam = 120                         # diversity, not depth, is the binding
                                       # constraint on this problem
    max_legs = 16
    hp_lo, hp_hi = 20.0, 4500.0        # km, altitude floor from the notebook's spec
    n_hp = 260                         # grid for the vectorised sweep
    n_probe = 24                       # probe density is NOT the bottleneck: from the
                                       # hand plan's own post-leg-6 state, 22 probes
                                       # already find its 1.46-day Io->Ganymede hop
                                       # exactly, and 300 add nothing.
    res_jmax = 3                       # resonances i:j with j <= this
    res_ratio = (1.5, 20.0)            # allowed i/j
    miss_tol = 6000.0                  # km; the hand plan runs up to ~4400
    max_leg_days = 26.0                # hand plan's longest leg is 24.8 d
    budget_day = 240.0                 # sim days. The mission's own '< 7 months from
                                       # capture epoch' requirement is ~day 227.
    coarse_s = 500.0                   # discovery time step
    goal_moon = 'Europa'
    goal_vinf = 2.0
    w_time = 1.5                       # Time is a real resource (at 0.4 the beam
                                       # burned 26 days on a 1.5-day leg), but at 3.0
                                       # it starves the ladder. budget_day still caps.
    w_gap = 4.0                        # km/s per RJ by which the orbit fails
                                       # to reach the goal moon's orbit
    w_way = 1.5                        # v_inf at the moons still to be flown by
                                       # (steering authority). Informs, does not rule.
    bin_rp, bin_ra, bin_day = 0.25, 1.0, 1.0     # dedupe bins (RJ, RJ, days).
                                       # KEEP bin_day FINE. Two nodes on the same
                                       # orbit days apart have completely different
                                       # moon phasing, and the Io ladder's real job is
                                       # phasing the Ganymede handoff -- a 10-day bin
                                       # merged them away and cost 3 km/s at arrival.
    fam_day = 1.0                      # arrival-day bucket for grouping probes
    n_refine = 18                      # golden iterations on hp per family


def _res_pairs(cfg):
    out = []
    for j in range(1, cfg.res_jmax + 1):
        for i in range(2, int(cfg.res_ratio[1] * j) + 1):
            if gcd(i, j) == 1 and cfg.res_ratio[0] <= i / j <= cfg.res_ratio[1]:
                out.append((i, j))
    return out


def _bisect(fun, lo, hi, n=60):
    flo = fun(lo)
    for _ in range(n):
        mid = .5 * (lo + hi)
        if flo * fun(mid) <= 0: hi = mid
        else: lo, flo = mid, fun(mid)
    return .5 * (lo + hi)


class Tour(Search):

    def score(self, a, e, t, cfg, remaining=()):
        """Cost-to-go, lower is better. Three ingredients.

        1. v_inf at the goal moon. The obvious term -- but on its own it is a
           MISLEADING greedy signal here. Measured at leg 8, an orbit of 6.60x17.50 RJ
           scores better on v_inf_Europa than the hand plan's 5.50x15.60 (6.15 vs
           6.85 km/s) and still loses the tour by 3 km/s at arrival.

        2. Steering authority at the moons still to be flown by. A flyby's turn angle
           is 2*asin(1/(1 + r_p*v_inf^2/mu)), so it GROWS as v_inf falls: arriving
           slowly at the endgame moon is what buys the orbit changes the last legs
           need. On that same leg-8 comparison the hand plan is ahead exactly here --
           v_inf at Ganymede 3.56 vs 4.32 km/s -- which is what its endgame spends.
           Weighted by how many of the remaining flybys happen at each moon.

        3. Reachability and elapsed time. Note the sqrt(|val|): the Tisserand
           expression goes negative when the orbit does not cross the moon's orbit at
           all, and an earlier version clamped that to zero -- scoring a craft that had
           sailed past Europa as a perfect tangent arrival, and returning zero valid
           tours. Never reintroduce the clamp.
        """
        a_m, v_m = self.ref[cfg.goal_moon]
        rp, ra = a * (1 - e), a * (1 + e)
        p = max(a * (1 - e * e), 0.0)
        val = 3.0 - a_m / a - 2.0 * np.sqrt(p / a_m)
        gap = (max(0.0, rp - a_m) + max(0.0, a_m - ra)) / self.RJ
        s = (v_m * np.sqrt(abs(val))
             + cfg.w_gap * gap
             + cfg.w_time * (t / DAY) / cfg.budget_day)
        if remaining and cfg.w_way:
            n_tot = len(remaining)
            for M in {lg[0] for lg in remaining} - {cfg.goal_moon}:
                n = sum(1 for lg in remaining if lg[0] == M)
                am, vm_ = self.ref[M]
                v = 3.0 - am / a - 2.0 * np.sqrt(p / am)
                s += cfg.w_way * (n / n_tot) * vm_ * np.sqrt(abs(v))
        return s

    def _mk(self, node, kind, tg, sign, hp, param, t_enc, r_new, v_new, cfg,
            remaining=()):
        """Propagate the post-flyby state to the encounter and build the child."""
        dt = t_enc - node.t
        rr, vv = kepler_fast(r_new, v_new, [dt], self.MU)
        rm, vm = self.moons[tg].state(t_enc)
        miss = float(np.linalg.norm(rr[0] - rm))
        if miss > cfg.miss_tol:
            return None
        vinf = float(np.linalg.norm(vv[0] - vm))
        a, e = elements(rr[0], vv[0], self.MU)
        a, e = float(a[0]), float(e[0])
        if not np.isfinite(a) or a <= 0 or e >= 1.0:
            return None
        leg = dict(kind=kind, at=node.at, tg=tg, sign=sign, hp=hp, param=param,
                   t_d=t_enc / DAY, miss=miss, vinf=vinf,
                   rp=a * (1 - e) / self.RJ, ra=a * (1 + e) / self.RJ)
        n = Node(rr[0], vv[0], t_enc, tg, node.legs + (leg,), a, e, vinf)
        n.score = self.score(a, e, t_enc, cfg, remaining)
        return n

    def expand(self, node, leg, cfg, remaining=()):
        """Children of `node` for ONE prescribed leg.

        `leg` is (at, target, sign), taken from the hand plan rather than searched, so
        there is no loop over signs and no loop over target moons -- the only freedom
        left is which resonance, or which encounter, to take.
        """
        at, tg, sign = leg[0], leg[1], leg[2]
        forced = leg[3] if len(leg) > 3 else None   # pin the resonance for this leg
        assert node.at == at, f"leg expects to depart {at}, node is at {node.at}"
        m = self.moons[at]
        rm, vm = m.state(node.t)
        hps = np.geomspace(cfg.hp_lo, cfg.hp_hi, cfg.n_hp)
        t_left = cfg.budget_day * DAY - node.t
        kids = []

        # one vectorised sweep of the altitude grid -> post-flyby elements
        vout = flyby_v(node.v, vm, m.mu, m.R, hps, sign)
        a, e = elements(np.repeat(rm[None, :], len(hps), 0), vout, self.MU)
        ell = np.isfinite(a) & (a > 0) & (e < 1.0)
        if not ell.any():
            return kids
        rp, ra = a * (1 - e), a * (1 + e)
        per = np.where(ell, 2 * np.pi * np.sqrt(np.abs(a) ** 3 / self.MU), np.nan)

        def period_of(h):
            vv = flyby_v(node.v, vm, m.mu, m.R, h, sign)[0]
            aa, _ = elements(rm, vv, self.MU)
            if not np.isfinite(aa[0]) or aa[0] <= 0:
                return np.nan
            return 2 * np.pi * np.sqrt(aa[0] ** 3 / self.MU)

        # ---- same-moon: integer resonances, read off the period-vs-altitude curve
        if at == tg:
            for (i, j) in ([forced] if forced else cfg._pairs):
                Pt = (i / j) * m.T
                tof = i * m.T
                if not forced and tof > min(t_left, cfg.max_leg_days * DAY * 1.5):
                    continue
                f = per - Pt
                ok = ell[:-1] & ell[1:] & np.isfinite(f[:-1]) & np.isfinite(f[1:])
                cross = np.nonzero(ok & (np.sign(f[:-1]) != np.sign(f[1:])))[0]
                for k in cross:
                    hp = _bisect(lambda h: period_of(h) - Pt, hps[k], hps[k + 1])
                    if not (cfg.hp_lo <= hp <= cfg.hp_hi):
                        continue
                    r_new, v_new = flyby_full(node.r, node.v, rm, vm, m.mu, m.R, hp, sign)
                    encs = self.encounters(r_new, v_new, node.t, at,
                                           tof - .5 * DAY, tof + .5 * DAY, 250.0,
                                           n_rounds=3)
                    if not encs:
                        continue
                    t_enc = min(encs, key=lambda x: x[1])[0]
                    kid = self._mk(node, 'res', at, sign, hp, (i, j),
                                   t_enc, r_new, v_new, cfg, remaining)
                    if kid: kids.append(kid)

        # ---- free encounters with the target: cross-moon transfers, and same-moon
        # ---- non-integer node-crossings (which is what the hand plan's leg 8 is)
        if forced:
            return kids
        t_hi = min(cfg.max_leg_days * DAY, t_left)
        if t_hi <= 0.5 * DAY:
            return kids
        a_t = self.ref[tg][0]
        cand = np.nonzero(ell & (rp <= a_t * 1.02) & (ra >= a_t * 0.98))[0]
        if len(cand) == 0:
            return kids
        pick = cand[np.linspace(0, len(cand) - 1,
                                min(cfg.n_probe, len(cand))).astype(int)]
        dts = np.arange(0.15 * DAY, t_hi, cfg.coarse_s)
        rm_all = self.moons[tg].states(node.t + dts)[0]      # computed once
        raw = []
        for idx in pick:
            r_new, v_new = flyby_full(node.r, node.v, rm, vm, m.mu, m.R,
                                      float(hps[idx]), sign)
            for (t_enc, miss) in self.encounters(
                    r_new, v_new, node.t, tg, 0.15 * DAY, t_hi,
                    cfg.coarse_s, rm_cache=rm_all, n_rounds=2):
                raw.append((float(hps[idx]), t_enc, miss))
        for hp0, dt0 in self._families(raw, node.t, cfg):
            hp_ref = self._refine_hp(node, m, rm, vm, tg, sign, hp0, dt0, cfg)
            if hp_ref is None:
                continue
            hp, t_e = hp_ref
            r_n, v_n = flyby_full(node.r, node.v, rm, vm, m.mu, m.R, hp, sign)
            kid = self._mk(node, 'xfer', tg, sign, hp,
                           (t_e - node.t) / DAY, t_e, r_n, v_n, cfg, remaining)
            if kid: kids.append(kid)
        return kids

    @staticmethod
    def _families(raw, t0, cfg):
        """Collapse (hp, t_enc, miss) probes into distinct encounter families.

        One geometric encounter shows up across a run of neighbouring hp probes;
        refining each separately is what made this unusably slow. Bucket by arrival
        day, keep each bucket's lowest-miss probe as the seed.
        """
        best = {}
        for hp, t_enc, miss in raw:
            if miss > cfg.miss_tol * 6:
                continue
            key = int((t_enc - t0) / DAY / cfg.fam_day)
            if key not in best or miss < best[key][2]:
                best[key] = (hp, t_enc, miss)
        return [(hp, t_enc - t0) for hp, t_enc, _ in best.values()]

    def _refine_hp(self, node, m, rm, vm, tg, sign, hp0, dt0, cfg, span=1.35):
        """Golden-section hp around hp0 to minimise the encounter miss."""
        w = 0.45 * DAY

        def enc(h):
            if not (cfg.hp_lo <= h <= cfg.hp_hi):
                return None
            r_n, v_n = flyby_full(node.r, node.v, rm, vm, m.mu, m.R, h, sign)
            es = self.encounters(r_n, v_n, node.t, tg,
                                 max(0.1 * DAY, dt0 - w), dt0 + w, 250.0, n_rounds=3)
            return min(es, key=lambda x: x[1]) if es else None

        lo, hi = max(cfg.hp_lo, hp0 / span), min(cfg.hp_hi, hp0 * span)
        for _ in range(cfg.n_refine):
            m1 = lo + (hi - lo) * .382; m2 = lo + (hi - lo) * .618
            e1, e2 = enc(m1), enc(m2)
            f1 = e1[1] if e1 else 1e12
            f2 = e2[1] if e2 else 1e12
            if f1 < f2: hi = m2
            else: lo = m1
        hp = .5 * (lo + hi)
        e = enc(hp)
        return (hp, e[0]) if e else None

    def prune(self, cand, cfg):
        best = {}
        for c in cand:
            key = (c.at,
                   int(c.rp / self.RJ / cfg.bin_rp),
                   int(c.ra / self.RJ / cfg.bin_ra),
                   int(c.t / DAY / cfg.bin_day))
            if key not in best or c.score < best[key].score:
                best[key] = c
        return sorted(best.values(), key=lambda n: n.score)[:cfg.beam]

    def run(self, root, sequence, cfg, verbose=True):
        """Walk the prescribed leg sequence. Every survivor at the end is a complete
        tour, so there is no goal test mid-search -- finishers are ranked afterwards."""
        cfg._pairs = _res_pairs(cfg)
        beam = [root]
        for depth, leg in enumerate(sequence):
            cand = []
            for n in beam:
                cand.extend(self.expand(n, leg, cfg, sequence[depth + 1:]))
            beam = self.prune(cand, cfg)
            if verbose:
                b = beam[0] if beam else None
                at, tg, sg = leg[0], leg[1], leg[2]
                head = (f"  leg {depth + 1:2d} {at:8s}->{tg:8s} sgn{sg:+d}: "
                        f"{len(cand):4d} cand -> beam {len(beam):3d}")
                print(head + (f" | best {b.rp / self.RJ:5.2f}x{b.ra / self.RJ:6.2f} RJ "
                              f"day {b.t / DAY:6.2f} vinf {b.vinf:6.3f}" if b else
                              "  <-- NO FEASIBLE CHILD, sequence dies here"),
                      flush=True)
            if not beam:
                return [], depth
        return beam, len(sequence)

## 4. Run it

`SEQUENCE` is the part taken from the hand plan. Every other knob lives on `Cfg`.

In [7]:
# A leg is (at, target, sign) to SEARCH it, or (at, target, sign, (i, j)) to PIN its
# resonance. at/target/sign come from the hand plan; everything else is derived.
#
# The six Io legs are pinned here. That is not cosmetic -- see the note below: the
# ladder is the one part of this plan a forward beam search cannot find, because v_inf
# at Io is invariant across every Io flyby, so no energy-based score can rank one
# ladder against another. Delete the (i, j) entries to watch it fail.
SEQUENCE = [
    ('Io', 'Io', -1, (14, 1)), ('Io', 'Io', -1, (9, 1)), ('Io', 'Io', -1, (6, 1)),
    ('Io', 'Io', -1, (13, 3)), ('Io', 'Io', -1, (7, 2)), ('Io', 'Io', -1, (8, 3)),
    ('Io',       'Ganymede', -1),
    ('Ganymede', 'Ganymede', +1),
    ('Ganymede', 'Europa',   -1),
    ('Europa',   'Ganymede', -1),
    ('Ganymede', 'Ganymede', +1),
    ('Ganymede', 'Europa',   +1),
]

# FastMoon swap, verified against the exact solver before use.
fast = {n: FastMoon(m, kepler_prop, MU) for n, m in moons_tb.items()}
_tp = np.linspace(0, 40 * DAY, 977)
for n in fast:
    dr = np.abs(moons_tb[n].states(_tp)[0] - fast[n].states(_tp)[0]).max()
    dv = np.abs(moons_tb[n].states(_tp)[1] - fast[n].states(_tp)[1]).max()
    print(f"FastMoon {n:9s} max dr {dr:.2e} km   max dv {dv:.2e} km/s")
    assert dr < 1e-3 and dv < 1e-6, f"FastMoon {n} disagrees with the exact solver"

search = Tour(fast, MU, RJ_KM)

# Root = the post-PRM state at the day-46 Io encounter, i.e. exactly cell 5's `orb`.
_r0 = orb_post_prm.r.to_value('km'); _v0 = orb_post_prm.v.to_value('km/s')
_t0 = (orb_post_prm.epoch - start_date).to_value('s')
_a0, _e0 = elements(_r0, _v0, MU); _a0, _e0 = float(_a0[0]), float(_e0[0])
_rm, _vm = fast['Io'].state(_t0)
root = Node(_r0, _v0, _t0, 'Io', (), _a0, _e0, float(np.linalg.norm(_v0 - _vm)))
cfg = Cfg()
root.score = search.score(_a0, _e0, _t0, cfg, SEQUENCE)
print(f"\nroot: day {_t0/DAY:.2f} at Io, {_a0*(1-_e0)/RJ_KM:.2f} x "
      f"{_a0*(1+_e0)/RJ_KM:.2f} RJ, vinf_Io {root.vinf:.3f} km/s")

FastMoon Io        max dr 2.33e-09 km   max dv 4.19e-10 km/s
FastMoon Europa    max dr 3.58e-09 km   max dv 2.55e-10 km/s
FastMoon Ganymede  max dr 5.24e-09 km   max dv 2.32e-10 km/s

root: day 46.00 at Io, 5.67 x 95.09 RJ, vinf_Io 7.491 km/s


In [8]:
_t_start = _t.time()
finishers, reached = search.run(root, SEQUENCE, cfg)
print(f"\nsearch took {_t.time() - _t_start:.1f} s; reached leg "
      f"{reached}/{len(SEQUENCE)}; {len(finishers)} complete tours")

  leg  1 Io      ->Io       sgn-1:    1 cand -> beam   1 | best  5.59x 62.98 RJ day  70.79 vinf  7.491
  leg  2 Io      ->Io       sgn-1:    1 cand -> beam   1 | best  5.50x 45.57 RJ day  86.73 vinf  7.491
  leg  3 Io      ->Io       sgn-1:    1 cand -> beam   1 | best  5.38x 33.60 RJ day  97.35 vinf  7.491
  leg  4 Io      ->Io       sgn-1:    1 cand -> beam   1 | best  5.25x 26.13 RJ day 120.37 vinf  7.491
  leg  5 Io      ->Io       sgn-1:    1 cand -> beam   1 | best  5.13x 22.08 RJ day 132.77 vinf  7.491
  leg  6 Io      ->Io       sgn-1:    1 cand -> beam   1 | best  4.95x 17.75 RJ day 146.93 vinf  7.491
  leg  7 Io      ->Ganymede sgn-1:    1 cand -> beam   1 | best  4.77x 15.20 RJ day 148.39 vinf  3.575
  leg  8 Ganymede->Ganymede sgn+1:    3 cand -> beam   3 | best  6.31x 16.09 RJ day 163.71 vinf  3.571
  leg  9 Ganymede->Europa   sgn-1:    9 cand -> beam   9 | best  9.20x 18.54 RJ day 184.10 vinf  2.753
  leg 10 Europa  ->Ganymede sgn-1:   12 cand -> beam  11 | best  7.24x 15

## 5. Result, side by side with the hand plan

In [9]:
HAND = [(183.6, 70.79, 7.491), (269.1, 86.73, 7.491), (53.0, 97.35, 7.491),
        (162.9, 120.37, 7.491), (881.0, 132.77, 7.491), (31.9, 146.93, 7.491),
        (534.8, 148.39, 3.575), (2823.0, 170.55, 3.571), (248.0, 178.00, 4.821),
        (568.1, 198.24, 2.158), (1106.3, 212.55, 2.158), (3998.7, 227.25, 1.702)]

if not finishers:
    print(f"sequence infeasible: no child at leg {reached + 1}")
else:
    finishers.sort(key=lambda n: (n.vinf, n.t))
    best = finishers[0]
    print(f"\n{'':21} {'-------- derived --------':>33}   {'----- hand plan -----':>26}")
    print(f"{'#':>2} {'at':9}->{'tgt':9} {'h_p km':>8} {'day':>7} {'vinf':>6} {'miss':>6} "
          f"{'leg':>11} | {'h_p km':>8} {'day':>7} {'vinf':>6}")
    print('-' * 104)
    for k, L in enumerate(best.legs):
        p = (f"{L['param'][0]}:{L['param'][1]}" if L['kind'] == 'res'
             else f"{L['param']:.2f}d")
        hh, hd, hv = HAND[k]
        print(f"{k+1:>2} {L['at']:9}->{L['tg']:9} {L['hp']:8.1f} {L['t_d']:7.2f} "
              f"{L['vinf']:6.3f} {L['miss']:6.0f} {L['kind']+' '+p:>11} | "
              f"{hh:8.1f} {hd:7.2f} {hv:6.3f}")
    print(f"\narrival   : {best.vinf:.3f} km/s on sim day {best.t/DAY:.2f}"
          f"   (hand plan 1.702 km/s on day 227.25)")
    print(f"min h_p   : {min(L['hp'] for L in best.legs):.1f} km  (constraint >= 20)")
    print(f"worst miss: {max(L['miss'] for L in best.legs):.0f} km")

    print("\n# paste straight into cell 6 of new_baseline_traj.ipynb")
    print("PLAN = [")
    for L in best.legs:
        prm = (f"({L['param'][0]}, {L['param'][1]})" if L['kind'] == 'res'
               else f"{L['param']:.2f}")
        print(f"    ({L['kind']!r:7}, {L['at']!r:11}, {L['tg']!r:11}, "
              f"{L['sign']:2d}, {L['hp']:7.1f}, {prm}),")
    print("]")


                              -------- derived --------        ----- hand plan -----
 # at       ->tgt         h_p km     day   vinf   miss         leg |   h_p km     day   vinf
--------------------------------------------------------------------------------------------------------
 1 Io       ->Io           183.6   70.79  7.491      0    res 14:1 |    183.6   70.79  7.491
 2 Io       ->Io           269.2   86.73  7.491      0     res 9:1 |    269.1   86.73  7.491
 3 Io       ->Io            53.1   97.35  7.491      0     res 6:1 |     53.0   97.35  7.491
 4 Io       ->Io           162.9  120.37  7.491      0    res 13:3 |    162.9  120.37  7.491
 5 Io       ->Io           881.0  132.77  7.491      0     res 7:2 |    881.0  132.77  7.491
 6 Io       ->Io            31.9  146.93  7.491      0     res 8:3 |     31.9  146.93  7.491
 7 Io       ->Ganymede     534.8  148.39  3.575   4394  xfer 1.46d |    534.8  148.39  3.575
 8 Ganymede ->Ganymede    2822.7  170.55  3.571    369 xfer 22.16

## What is, and is not, re-derivable

Pinning the six Io resonances, the beam re-derives **every other number in the table**
— all twelve flyby altitudes, every encounter time, the whole Ganymede endgame and the
Europa peri-raise — to 3–4 significant figures in about 25 seconds, arriving at
**1.701 km/s on sim day 227.25** against the published 1.702 on 227.25.

Unpin the ladder and it fails. Four tuned runs gave 7.224, 4.687, 6.661 and 5.966 km/s.
The reason is structural, not a matter of weights: **$v_\infty$ at Io is invariant
across all seven Io flybys** (7.491 km/s throughout), so no energy-based objective can
rank one ladder against another. The ladder's real job is *phasing* — it manoeuvres the
craft so that Ganymede sits **1.46 days** away at the sixth Io flyby — and that payoff
is invisible until leg 7 is attempted.

That the machinery is sound was checked separately: handed the hand plan's own
post-leg-6 state, `expand()` reproduces leg 7 exactly (h_p 534.8 km, day 148.39,
$v_\infty$ 3.575 km/s) at the *lowest* probe density, and 300 probes add nothing.

So the plan decomposes into three layers: the **moon sequence** (a human Tisserand-graph
decision), the **Io resonance ladder** (found by exhaustive search or by reasoning
backwards from the handoff — not by forward greedy search), and **everything else**,
which follows mechanically from those two.

### Other caveats

* **Same model as the baseline** — patched conics, instantaneous flybys rotating
  $v_\infty$ about the ecliptic normal, coplanar two-body moons from Horizons
  osculating states. This re-derives a plan *within* that model; it does not validate it.
* **The heuristic is optimistic where it matters most.** On the final Europa leg it
  reads 1.49 km/s against a propagated 1.702 — a 12% underestimate right where the
  < 2 km/s constraint bites. It is used only to *rank*; every reported number comes
  from the propagation.
* **Beam search is not optimal**, and gives no guarantee the best tour was found.
* **The PRM burn is fixed** at cell 5's 814.7 m/s and day-46 Io arrival.
